# Installing Libraries

In [1]:
!pip install accelerate peft bitsandbytes transformers trl -q


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


# Imports

In [9]:
# load the required packages.
import torch
from datasets import load_dataset, Dataset
from peft import LoraConfig, AutoPeftModelForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer
import os

# Paths

In [10]:
dataset="cleaned_movie.csv"
model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_model="tinyllama-colorist-v1"

In [16]:
from datasets import Dataset
import pandas as pd

def prepare_train_data(dataset_path):
    df = pd.read_csv(dataset_path)

    required_columns = [
        'movie_id', 'title', 'cast', 'crew', 'overview', 'budget', 'genres',
        'homepage', 'keywords', 'original_language', 'popularity',
        'production_companies', 'production_countries', 'release_date',
        'revenue', 'runtime', 'spoken_languages', 'status', 'tagline',
        'vote_average', 'vote_count'
    ]
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' missing from the dataset.")

    # Convert object/list columns to string
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].apply(lambda x: str(x))

    # Combine all columns into a single 'text' column
    df['text'] = df.apply(lambda row: "\n".join([f"{col}: {row[col]}" for col in df.columns if col != 'text']), axis=1)

    return Dataset.from_pandas(df)

# Usage
data = prepare_train_data("cleaned_movie.csv")


In [17]:
print(data)

Dataset({
    features: ['movie_id', 'title', 'cast', 'crew', 'overview', 'budget', 'genres', 'homepage', 'keywords', 'original_language', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'vote_average', 'vote_count', 'text'],
    num_rows: 3459
})


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

def get_model_and_tokenizer(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token  


    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,   
        device_map="auto"
    )

    model.config.use_cache = False  
    return model, tokenizer

model, tokenizer = get_model_and_tokenizer(model_id)


In [30]:
model, tokenizer = get_model_and_tokenizer(model_id)

In [31]:
peft_config = LoraConfig(
        r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )

In [ ]:
training_arguments = TrainingArguments(
        output_dir=output_model,
        per_device_train_batch_size=16,
        gradient_accumulation_steps=4,
        optim="paged_adamw_32bit",
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        save_strategy="epoch",
        logging_steps=10,
        num_train_epochs=3,
        max_steps=250,
        fp16=True,
    )

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=data,
    peft_config=peft_config,
    args=training_arguments
)


Adding EOS to train dataset:   0%|          | 0/3459 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3459 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (19034 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/3459 [00:00<?, ? examples/s]

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [34]:
trainer.train()

Step,Training Loss
10,0.769600
20,0.699500
30,0.646100
40,0.621000
50,0.584800
60,0.582900
70,0.572200
80,0.565200
90,0.560700
100,0.560700


TrainOutput(global_step=250, training_loss=0.5788164596557617, metrics={'train_runtime': 867.5149, 'train_samples_per_second': 18.443, 'train_steps_per_second': 0.288, 'total_flos': 1.0025487163824538e+17, 'train_loss': 0.5788164596557617})

In [35]:
from peft import AutoPeftModelForCausalLM, PeftModel
from transformers import AutoModelForCausalLM
import torch
import os

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, load_in_8bit=False,device_map="auto",trust_remote_code=True)

model_path = "tinyllama-colorist-v1/checkpoint-250"

peft_model = PeftModel.from_pretrained(model, model_path, from_transformers=True, device_map="auto")

model = peft_model.merge_and_unload()

In [36]:
def formatted_prompt(question)-> str:
    return f"<|user|>\n{question}</s>\n<|assistant|>"

In [37]:
from transformers import GenerationConfig
from time import perf_counter

def generate_response(user_input):

  prompt = formatted_prompt(user_input)

  inputs = tokenizer([prompt], return_tensors="pt")
  generation_config = GenerationConfig(penalty_alpha=0.6,do_sample = True,
      top_k=5,temperature=0.5,repetition_penalty=1.2,
      max_new_tokens=12,pad_token_id=tokenizer.eos_token_id
  )
  start_time = perf_counter()

  inputs = tokenizer(prompt, return_tensors="pt").to('cuda')

  outputs = model.generate(**inputs, generation_config=generation_config)
  print(tokenizer.decode(outputs[0], skip_special_tokens=True))
  output_time = perf_counter() - start_time
  print(f"Time taken for inference: {round(output_time,2)} seconds")

In [38]:
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, load_in_8bit=False,
                                             device_map="auto", trust_remote_code=True)

model_path = "tinyllama-colorist-v1/checkpoint-250"

peft_model = PeftModel.from_pretrained(model, model_path, from_transformers=True, device_map="auto")
model = peft_model.merge_and_unload()


In [39]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token


In [40]:
def formatted_prompt(question) -> str:
    return f"<|user|>\n{question}</s>\n<|assistant|>"


In [41]:
from transformers import GenerationConfig
from time import perf_counter
import torch

def generate_response(user_input):
    prompt = formatted_prompt(user_input)

    # Move inputs to GPU
    inputs = tokenizer(prompt, return_tensors="pt").to('cuda')

    generation_config = GenerationConfig(
        penalty_alpha=0.6,
        do_sample=True,
        top_k=5,
        temperature=0.5,
        repetition_penalty=1.2,
        max_new_tokens=50,   # Increase so answers are not cut off
        pad_token_id=tokenizer.eos_token_id
    )

    start_time = perf_counter()

    outputs = model.generate(**inputs, generation_config=generation_config)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    output_time = perf_counter() - start_time

    print(response)
    print(f"Time taken for inference: {round(output_time,2)} seconds")


In [ ]:
generate_response("Give me some good movie recommendations")


<|user|>
Give me some good movie recommendations 
<|assistant|>
1. The Lord of the Rings Trilogy (2001-2003) - This epic fantasy trilogy is a must-watch for any fan of J.R.R. Tolkien
Time taken for inference: 1.43 seconds


In [44]:
generate_response("who are the actors of Lord of the Rings Trilogy")

<|user|>
who are the actors of Lord of the Rings Trilogy 
<|assistant|>
The cast and crew for Peter Jackson's The Lord of the Rings trilogy included:

1. Ian McKellen as Gandalf the Grey
2. Viggo Mortensen as Aragorn
3. Sean Ast
Time taken for inference: 1.36 seconds


In [ ]:
if __name__ == "__main__":
    print("Welcome to the Movie Chatbot! Type 'exit' to quit.")
    while True:
        user_input = input("You: ")
        if user_input.lower() in ["exit", "quit"]:
            break
        generate_response(user_input)

Welcome to the Movie Chatbot! Type 'exit' to quit.
<|user|>
Can you recommend a good sad movie 
<|assistant|>
I recommend 'The Last Temptation of Christ'. The film is based on the novel "Jesus of Nazareth", written by Jean-Pierre Melville.
Time taken for inference: 1.48 seconds
<|user|>
Give me a brief overview of Toy Story 3 
<|assistant|>
Film about an AI toy named Woody, who gets separated from his owner and ends up in the care of a young boy. After being abandoned by his girlfriend, he is adopted by a new family with a child named B
Time taken for inference: 3.52 seconds
<|user|>
What genre is Jurrasic World 
<|assistant|>
Jurrasic World belongs to the ['Action', 'Adventure'] genre.
Time taken for inference: 0.77 seconds


In [ ]:
# 427606fdc777fd9ff250a0ec64921fc5d739c069

In [ ]:
from datasets import Dataset
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# -----------------------
# Prepare data as context
# -----------------------
def load_movie_dataset(dataset_path):
    data_df = pd.read_csv(dataset_path)

    required_columns = ['movie_id', 'title', 'cast', 'crew', 'overview', 'budget', 'genres',
       'homepage', 'keywords', 'original_language', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline',
       'vote_average', 'vote_count']
    for col in required_columns:
        if col not in data_df.columns:
            raise ValueError(f"Column '{col}' is missing from the dataset.")

    return data_df

# -----------------------
# Load model + tokenizer
# -----------------------
def get_model_and_tokenizer(model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto"
    )

    return model, tokenizer

# -----------------------
# Generate responses
# -----------------------
def generate_response(user_query, data_df, model, tokenizer):
    # Use dataset context (small subset for efficiency)
    sampled_data = data_df.sample(min(5, len(data_df)))  # pick 5 random movies
    context = "\n".join(
        [f"Title: {row['title']}\nOverview: {row['overview']}\nGenre: {row['genres']}" 
         for _, row in sampled_data.iterrows()]
    )

    prompt = f"<|user|>\nContext:\n{context}\n\nQuestion: {user_query}\n<|assistant|>\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        generation_config=GenerationConfig(
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9
        )
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip out the prompt part to show only answer
    answer = response.split("<|assistant|>")[-1].strip()
    print(f"<|assistant|>\n{answer}")
    

# -----------------------
# Main loop
# -----------------------
if __name__ == "__main__":
    data_df = load_movie_dataset("cleaned_movie.csv")
    model, tokenizer = get_model_and_tokenizer()

    print("Welcome to the Movie Chatbot! Type 'exit' to quit.")
    while True:
        user_input = input("<|user|>\n")
        if user_input.lower() in ["exit", "quit"]:
            break
        generate_response(user_input, data_df, model, tokenizer)


Welcome to the Movie Chatbot! Type 'exit' to quit.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|assistant|>
Sure, here's a list of movies that fit the criteria you provided:

1. "Deck the Halls" (2006) - This holiday-themed comedy features a determined husband who tries to unseat his wife's rival in the town's holiday decorating competition.

2. "Harry Brown" (2009) - This British thriller follows an elderly ex-serviceman and widower who looks to avenge his best friend's murder by doling out his own form of justice.

3. "Riddick" (2013) - This sci-fi action movie follows a man who is left for dead on a desolate planet and becomes more powerful and dangerous than ever before.

4. "Night of the Living Dead" (1968) - This classic horror film follows a group of people trapped inside a


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|assistant|>
Harry Brown was released in 2009.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|assistant|>
Harry Brown is a 2009 British crime drama film directed by Daniel Barber and starring Daniel Craig, Ciarán Hinds, and Rachel Weisz. The film tells the story of Harry Brown, a retired British soldier who becomes a vigilante after his wife is killed by a gang of drug dealers. The film is 112 minutes long.


In [3]:
import pandas as pd
df2 = pd.read_csv("cleaned_movie.csv")
display(df2)

,movie_id,title,cast,crew,overview,budget,genres,homepage,keywords,original_language,...,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,vote_average,vote_count
0,19995,Avatar,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...","In the 22nd century, a paraplegic Marine is di...",237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,...,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,7.2,11800
1,285,Pirates of the Caribbean: At World's End,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...","Captain Barbossa, long believed to be dead, ha...",300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",6.9,4500
2,206647,Spectre,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",A cryptic message from Bond’s past sends him o...,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,...,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,6.3,4466
3,49026,The Dark Knight Rises,"[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",Following the death of District Attorney Harve...,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,...,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,7.6,9106
4,49529,John Carter,"[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...","John Carter is a war-weary, former military ca...",260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,...,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",6.1,2124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3454,39851,Clean,"[{""cast_id"": 1, ""character"": ""Emily Wang"", ""cr...","[{""credit_id"": ""52fe47369251416c9106db9b"", ""de...","Tormented by a past life, garbage man Clean at...",0,"[{""id"": 18, ""name"": ""Drama""}]",NaN,"[{""id"": 6782, ""name"": ""addiction""}, {""id"": 155...",en,...,[],"[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2004-09-01,0,111.0,"[{""iso_639_1"": ""cn"", ""name"": ""\u5e7f\u5dde\u8b...",Released,"When you don't have a choice, you change.",6.7,17
3455,13898,The Circle,"[{""cast_id"": 3, ""character"": ""Nargess"", ""credi...","[{""credit_id"": ""52fe45b09251416c7

# RAG

In [ ]:
from datasets import Dataset
import pandas as pd
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# -----------------------
# Prepare data as context
# -----------------------
def load_movie_dataset(dataset_path):
    data_df = pd.read_csv(dataset_path)

    required_columns = ['movie_id', 'title', 'cast', 'crew', 'overview', 'budget', 'genres',
       'homepage', 'keywords', 'original_language', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline',
       'vote_average', 'vote_count']
    for col in required_columns:
        if col not in data_df.columns:
            raise ValueError(f"Column '{col}' is missing from the dataset.")

    return data_df

# -----------------------
# Load model + tokenizer
# -----------------------
def get_model_and_tokenizer(model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0"):
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",   # bf16/fp16 depending on GPU
        device_map="auto"     # uses all available GPUs
    )

    return model, tokenizer

# -----------------------
# Generate responses
# -----------------------
def generate_response(user_query, data_df, model, tokenizer):
    # Use dataset context (small subset for efficiency)
    sampled_data = data_df.sample(min(5, len(data_df)))  # pick 5 random movies
    context = "\n".join(
        [f"Title: {row['title']}\nOverview: {row['overview']}\nGenre: {row['genres']}" 
         for _, row in sampled_data.iterrows()]
    )

    prompt = f"<|user|>\nContext:\n{context}\n\nQuestion: {user_query}\n<|assistant|>\n" #system = "You are a movie expert"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start_time = time.time()
    outputs = model.generate(
        **inputs,
        generation_config=GenerationConfig(
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9
        )
    )
    end_time = time.time()

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = response.split("<|assistant|>")[-1].strip()

    print(f"<|assistant|>\n{answer}")
    print(f"Time taken for inference: {end_time - start_time:.2f} seconds\n")

# -----------------------
# Initialize once
# -----------------------
if __name__ == "__main__":
    data_df = load_movie_dataset("cleaned_movie.csv")
    model, tokenizer = get_model_and_tokenizer()


In [50]:
data_df = load_movie_dataset("cleaned_movie.csv")


In [51]:
generate_response("What is the best movie", data_df, model, tokenizer)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|assistant|>
The best movie is "Love the Coopers" based on the context provided.
Time taken for inference: 2.08 seconds



In [52]:
generate_response("Who are the actors in Love the Coopers", data_df, model, tokenizer)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|assistant|>
The actors in Love the Coopers are:

1. Diane Lane
2. John Goodman
3. Catherine Keener
4. Jeff Daniels
5. Marisa Tomei
6. Anna Faris
7. Bruce Willis
8. Sarah Silverman
9. Alan Arkin
10. Danny DeVito
11. Jon Favreau
12. Michael Douglas
13. Emma Stone
14. Reese Witherspoon
15. Martha Plimpton
16. David Spade
17. Steve Carell
18. Hugh Laurie
19. Ed Helms
20. Mindy Kaling
21. Anna Kendrick
22. Maya Rudolph
23. Leslie Mann
24. Tina Fey
25. Chris Rock
26. Andy Samberg
27.
Time taken for inference: 5.79 seconds



# BenchMark Sets

In [ ]:
# 100 Questions
# Accuracy
# Question | Ground Truth ->Ideal Answer -> Accuracy(Correct/Incorrect) --> Latency

# Crew of interstellar -> 

# Unsloth 
# QLORA